# 06 · The similarity functions in action

**Purpose.** Show what each function in `src/indurent_matching/similarity.py` does, by calling it on the real data.
M3 (Spitfire Park, Unit 42) is the running example; M2 and M5 show what "not available" looks like.

The choices behind the functions are in notebooks 02-04; how they answer prompt 1 is in notebook 05. This notebook
grows as the size and combining functions are added.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from indurent_matching.data import load_marketed, load_portfolio
from indurent_matching.similarity import (
    age_homogeneity,
    age_similarity,
    get_year_band,
    halving,
    has_usable_location,
    location_context,
    location_homogeneity,
    location_similarity,
    sits_on_estate,
    usable,
)

# Store each chart's data in the notebook itself; JupyterLab draws it with the Plotly extension.
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", None)  # show every column of wide tables

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")


**The data: cleaned, not raw.** The first five rows of each file as returned by `load_portfolio` and
`load_marketed`: columns renamed to snake_case (`size_sqm` becomes `size_sqft`), placeholders and zeros blanked,
ranges turned into midpoints, and location checks, estate structure and quality flags added (the `_doubtful` flags
and `quality_notes` are at the end). No value is filled in or overwritten; the raw spreadsheets are shown in
`01_eda.ipynb`.

In [2]:
portfolio.head()

,property_id,industrial_estate_name,unit_name,region,latitude,longitude,car_parking_spaces,size_sqft,build_year,yard_depth_m,min_eaves_m,max_eaves_m,doors,epc_rating,estate_units,is_single_unit,km_offshore,km_to_region,location_confidence,derived_region,nearest_city,miles_to_city,doors_describe_unit,doors_per_unit_estimate,size_sqft_doubtful,build_year_doubtful,yard_depth_m_doubtful,min_eaves_m_doubtful,max_eaves_m_doubtful,doors_doubtful,quality_notes
property_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,Ivory Wharf 66,Unit 1,Midlands,51.927270,-0.987484,22.0,7508.729478,1999.0,NaN,4.0,5.2,NaN,NaN,18,False,0.0,11.752692,medium,South East,Coventry,40.081020,True,NaN,False,False,False,False,False,False,[location medium confidence: 12 km outside lab...
2,2,Ivory Wharf 66,Unit 2,Midlands,51.928783,-0.988291,22.0,13612.939196,1975.0,NaN,4.0,5.2,NaN,NaN,18,False,0.0,11.635858,medium,South East,Coventry,39.975153,True,NaN,False,False,False,False,False,False,[location medium confidence: 12 km outside lab...
3,3,Ivory Wharf 66,Unit 3,Midlands,51.928141,-0.987723,22.0,4065.156838,1983.0,NaN,4.0,5.2,NaN,NaN,18,False,0.0,11.677409,medium,South East,Coventry,40.025469,True,NaN,False,False,False,False,False,False,[location medium confidence: 12 km outside lab...
4,4,Ivory Wharf 66,Unit 4,Midlands,51.927537,-0.987248,22.0,1421.273392,1998.0,NaN,4.0,5.2,NaN,NaN,18,False,0.0,11.718921,medium,South East,Coventry,40.071387,True,NaN,False,False,False,False,False,False,[location medium confidence: 12 km outside lab...
5,5,Ivory Wharf 66,Unit 5,Midlands,51.927563,-0.987496,22.0,2857.252395,1993.0,NaN,4.0,5.2,NaN,NaN,18,False,0.0,11.724978,medium,South East,Coventry,40.063996,True,NaN,False,False,False,False,False,False,[location medium confidence: 12 km outside lab...


In [3]:
marketed.head()

,listing_id,industrial_estate_name,unit_name,region,latitude,longitude,car_parking_spaces,size_sqft,build_year,yard_depth_m,min_eaves_m,max_eaves_m,doors,epc_rating,km_offshore,km_to_region,location_confidence,derived_region,nearest_city,miles_to_city,doors_describe_unit,doors_per_unit_estimate,size_sqft_doubtful,build_year_doubtful,yard_depth_m_doubtful,min_eaves_m_doubtful,max_eaves_m_doubtful,doors_doubtful,quality_notes
listing_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
M1,M1,Cherry Lane,Unit 10,North West,52.467186,-3.784003,30.0,16860.130016,1977.0,NaN,NaN,NaN,NaN,NaN,0.000000,87.347822,medium,Wales,Cardiff,72.802078,True,NaN,False,False,False,False,False,False,[location medium confidence: 87 km outside lab...
M2,M2,Tech Hub,Unit 8,South East,NaN,NaN,23.0,905.523948,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unknown,None,NaN,NaN,True,NaN,False,False,False,False,False,False,"[no coordinates, so location can't be used, no..."
M3,M3,Spitfire Park,Unit 42,Midlands,53.505831,-1.905054,49.0,2294.498110,1976.0,NaN,NaN,NaN,NaN,B,0.000000,0.000000,high,Midlands,Manchester,13.982244,True,NaN,False,False,False,False,False,False,[]
M4,M4,Stable Lane,Unit 14,South East,52.078733,-0.349448,48.0,617.297521,2000.0,NaN,7.00,7.00,16.0,C,0.000000,15.492603,medium,East of England,London,40.596202,True,NaN,False,False,False,False,False,True,[location medium confidence: 15 km outside lab...
M5,M5,Chancery Depot,Unit 2,NaN,52.485703,-4.478974,22.0,16613.786473,1966.0,NaN,5.21,5.46,3.0,A,27.274229,NaN,unknown,None,NaN,NaN,True,NaN,False,False,False,False,False,False,"[coordinates are 27 km offshore, so location c..."


**The running example: M3.** The fields the age and location functions use.

In [4]:
m3 = marketed.loc["M3"]
m3[["industrial_estate_name", "unit_name", "build_year", "size_sqft", "latitude", "longitude", "location_confidence"]]

industrial_estate_name    Spitfire Park
unit_name                       Unit 42
build_year                       1976.0
size_sqft                    2294.49811
latitude                      53.505831
longitude                     -1.905054
location_confidence                high
Name: M3, dtype: object

## 1. Age

**`age_homogeneity(listing, portfolio)`** (prompt 2): units in the listing's 5-year band / units in the most
frequent band. The bands come from **`get_year_band(year)`**, which gives the first year of the band a year falls
in:

In [5]:
pd.DataFrame({
    "year": [1975, 1976, 1979, 1980, 2022],
    "get_year_band(year)": get_year_band(pd.Series([1975, 1976, 1979, 1980, 2022])),
})

,year,get_year_band(year)
0,1975,1975
1,1976,1975
2,1979,1975
3,1980,1980
4,2022,2020


1975-1979 all fall in the band starting 1975; 1980 starts the next one. Counting the portfolio's usable build
years by band gives the score:

In [6]:
bands = get_year_band(usable(portfolio, "build_year").dropna()).value_counts().sort_index()
print(f"Units per band: 1975-79 has {bands[1975]}, the most frequent band ({bands.idxmax():.0f}-{bands.idxmax() + 4:.0f}) "
      f"has {bands.max()}, so M3 (1976) scores {bands[1975]} / {bands.max()} = {bands[1975] / bands.max():.2f}")
pd.DataFrame({
    "build_year": marketed["build_year"],
    "age_homogeneity": [round(age_homogeneity(row, portfolio), 2) for _, row in marketed.iterrows()],
}, index=marketed.index)

Units per band: 1975-79 has 109, the most frequent band (1995-1999) has 127, so M3 (1976) scores 109 / 127 = 0.86


,build_year,age_homogeneity
listing_id,,
M1,1977.0,0.86
M2,NaN,NaN
M3,1976.0,0.86
M4,2000.0,0.86
M5,1966.0,0.20


* M3's score comes straight from two counts: its band and the most frequent band.
* M1, M3 and M4 all score 0.86; M5 (1966) 0.20; **M2 has no build year, so NaN** ("not available").

**`age_similarity(listing, portfolio)`** (prompt 1): one score per portfolio unit. It turns the gap between build
years into a score with **`halving(gap, half_life)`**, the formula every pairwise score uses: 1 at a gap of 0,
halving with every `half_life` (10 years for age):

In [7]:
gaps = pd.Series([0, 5, 10, 20, 40, -10])
pd.DataFrame({"gap": gaps, "halving(gap, 10)": halving(gaps, 10).round(3)})

,gap,"halving(gap, 10)"
0,0,1.000
1,5,0.707
2,10,0.500
3,20,0.250
4,40,0.062
5,-10,0.500


0.5 at one half-life, 0.25 at two, and the same for -10 as for +10: direction doesn't matter. Applied to the gap
between M3 (1976) and every portfolio unit:

In [8]:
m3_age = age_similarity(m3, portfolio)
print(f"{len(m3_age):,} scores, one per portfolio unit; {m3_age.isna().sum()} are NaN "
      "(units with no usable build year)")

fig = px.scatter(x=portfolio["build_year"], y=m3_age, height=360, opacity=0.4,
                 title="Age similarity of every portfolio unit to M3 (built 1976)",
                 labels={"x": "Portfolio unit build year", "y": "Age similarity to M3"})
fig.add_vline(x=1976, line_dash="dash", line_color="firebrick")
fig.show()

1,250 scores, one per portfolio unit; 16 are NaN (units with no usable build year)


In [9]:
sample = portfolio.loc[[*m3_age.dropna().sort_values().index[[0, 300, 700, 1000, -1]]],
                       ["industrial_estate_name", "unit_name", "build_year"]]
sample["gap (years)"] = (sample["build_year"] - 1976).abs()
sample["age_similarity to M3"] = m3_age[sample.index].round(2)
sample

,industrial_estate_name,unit_name,build_year,gap (years),age_similarity to M3
property_id,,,,,
895,Sable Crescent 13,Unit 2,2022.0,46.0,0.04
1210,Umber Mews 9,Unit 1,2009.0,33.0,0.10
552,Teal Mews 69,Unit 3,1993.0,17.0,0.31
682,Ruby Junction 67,Unit 16,1982.0,6.0,0.66
969,Topaz Park 9,Unit 3,1976.0,0.0,1.00


* Every unit sits on one curve: 1 for units built in 1976, falling either side, and 0.5 at 1966 and 1986.
* The 16 units without a build year are NaN. They're left as NaN here; when the
  blocks are combined for the ranking, a unit's NaN counts as 0.
* For M2 every score would be NaN, because M2 has no build year.

## 2. Location

**`has_usable_location`** and **`sits_on_estate`** first: which listings can be placed, and which estate each
sits on. (Location quality is a confidence level rather than a doubtful flag, so it has its own helper instead of
`usable`.)

In [10]:
pd.DataFrame({
    "location_confidence": marketed["location_confidence"],
    "has_usable_location": has_usable_location(marketed),
    "sits_on_estate": [sits_on_estate(row, portfolio) for _, row in marketed.iterrows()],
})

,location_confidence,has_usable_location,sits_on_estate
listing_id,,,
M1,medium,True,Scarlet Pavilion 78
M2,unknown,False,NaN
M3,high,True,Velvet Park 60
M4,medium,True,Sable Mill 91
M5,unknown,False,NaN


* M1, M3 and M4 have usable locations, and each sits on an existing Indurent estate (its nearest portfolio unit
  is within 0.25 miles).
* M2 (no coordinates) and M5 (offshore) can't be placed, so they sit on nothing.

**`location_homogeneity`** (prompt 2) and **`location_context`** (the facts reported alongside it).

In [11]:
rows = []
for listing_id, row in marketed.iterrows():
    ctx = location_context(row, portfolio)
    rows.append({"listing": listing_id, "location_homogeneity": location_homogeneity(row, portfolio), **ctx})
pd.DataFrame(rows).set_index("listing").round(2)

,location_homogeneity,sits_on_estate,nearest_other_estate,nearest_other_estate_miles,better_placed_than_pct
listing,,,,,
M1,0.23,Scarlet Pavilion 78,Quartz Wharf 16,4.42,6.62
M2,NaN,NaN,NaN,NaN,NaN
M3,0.62,Velvet Park 60,Jade Bridge 6,9.12,62.91
M4,0.73,Sable Mill 91,Sable Beacon 12,6.33,85.43
M5,NaN,NaN,NaN,NaN,NaN


* **M3 scores 0.62**: its 5 nearest other estates (Velvet Park 60, the one it sits on, left out) are on average
  1.6 times further away than for the best-placed estate in the portfolio.
* The context gives the readable facts: M3 sits on Velvet Park 60, the next nearest estate is Jade Bridge 6, 9.1
  miles away, and M3 is better placed than 63% of Indurent's estates.
* M2 and M5 get NaN and None throughout.

**`location_similarity(listing, portfolio)`** (prompt 1): one score per portfolio unit. It reuses `halving`, with
10 miles in place of 10 years.

In [12]:
m3_location = location_similarity(m3, portfolio)
print(f"{m3_location.notna().sum()} units scored; {m3_location.isna().sum()} NaN (no usable location). "
      f"Units scoring 0.5 or more (within 10 miles): {(m3_location >= 0.5).sum()}")

located = portfolio.assign(similarity=m3_location).dropna(subset=["similarity"])
fig = px.scatter_geo(located, lat="latitude", lon="longitude", color="similarity", color_continuous_scale="Viridis",
                     hover_name="industrial_estate_name", height=650,
                     title="Location similarity of every portfolio unit to M3 (star)")
fig.update_traces(marker={"size": 6})
fig.add_scattergeo(lat=[m3["latitude"]], lon=[m3["longitude"]], name="M3", showlegend=False,
                   marker={"symbol": "star", "size": 16, "color": "firebrick"})
fig.update_geos(resolution=50, showland=True, landcolor="#f2f2f2", showcountries=True,
                lataxis_range=[49.8, 59], lonaxis_range=[-7, 2.2])
fig.show()

892 units scored; 358 NaN (no usable location). Units scoring 0.5 or more (within 10 miles): 46


In [13]:
nearest = located.sort_values("similarity", ascending=False)
by_estate = nearest.groupby("industrial_estate_name", sort=False).agg(
    units=("unit_name", "size"), similarity=("similarity", "max")).head(6)
by_estate["miles"] = (-10 * np.log2(by_estate["similarity"])).round(1)  # back from similarity to miles
by_estate.round(2)

,units,similarity,miles
industrial_estate_name,,,
Velvet Park 60,41,1.00,0.0
Jade Bridge 6,4,0.53,9.1
Indigo Wharf 90,1,0.51,9.6
Azure Summit 81,1,0.34,15.4
Scarlet Arc 73,1,0.33,16.2
Cedar Pavilion 55,8,0.29,17.7


* Units near M3 score close to 1 and the score fades with distance; most of the portfolio is far enough away to
  score near 0.
* The **41 units of Velvet Park 60**, the estate M3 sits on, all score about 1. That's why, without a cap, M3's
  top 10 for prompt 1 can be all Velvet Park 60: on location alone they're unbeatable.
* The next estates (Jade Bridge 6 at about 9 miles, and so on) score around 0.5.
* Units with no usable location are NaN here, and count as 0 when the blocks are combined.